In [2]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [1]:
from datasets import load_dataset

# Load IMDB dataset
dataset = load_dataset("Vinod-IE/IDMBreview")

# View the IMDB dataset
print(dataset)
print(dataset['train'][0])  # See what a sample looks like

README.md:   0%|          | 0.00/501 [00:00<?, ?B/s]

data/train-00000-of-00001-0aee48ca1cebf9(…):   0%|          | 0.00/3.09M [00:00<?, ?B/s]

data/validation-00000-of-00001-f845736b3(…):   0%|          | 0.00/355k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3592 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'review_length'],
        num_rows: 3592
    })
    validation: Dataset({
        features: ['review', 'sentiment', 'review_length'],
        num_rows: 400
    })
})
{'review': "I've seen this movie on several different occasions. I find one of the funniest things to do is to just watch the reactions of the different types of people who go to see it.<br /><br />Type 1: OLD PEOPLE. A lot of old Japanese men and women go to this movie because they think it will be a honest-to-goodness samurai movie with lots of swordplay and medieval Japanese dialogue. As soon the two protagonists begin debating horror movies while inserting expletives almost randomly throughout their sentences, the old people walk out, usually disgusted.<br /><br />Type 2: FILM SNOBS. These people think that just because a movie bears the label of Independent that it will automatically be a load of hard-to-follow, overemotional crap that may or

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load a small, fast model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Tokenize the text
def tokenize_function(examples):
    return tokenizer(examples['review'], padding='max_length', truncation=True, max_length=512)

# Apply to dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Map string sentiment labels to integers (0 for negative, 1 for positive)
def map_labels(examples):
    return {'labels': 0 if examples['sentiment'] == 'negative' else 1}

tokenized_datasets = tokenized_datasets.map(map_labels)

# Remove original 'sentiment' column and 'review_length' if not needed
tokenized_datasets = tokenized_datasets.remove_columns(['sentiment', 'review_length'])

# Create smaller subsets for faster training
small_train = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))
small_test = tokenized_datasets['validation'].shuffle(seed=42).select(range(200))

Map:   0%|          | 0/3592 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/3592 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [5]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate # Import the evaluate library

# Define how to compute metrics
metric = evaluate.load("accuracy") # Use evaluate.load()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Set training parameters
training_args = TrainingArguments(
    output_dir="./results",
    # Removed evaluation_strategy as it caused a TypeError
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

# Train!
trainer.train()

model.save_pretrained("./fine_tuned_sentiment_model")
tokenizer.save_pretrained("./fine_tuned_sentiment_model")


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_sentiment_model/tokenizer_config.json',
 './fine_tuned_sentiment_model/tokenizer.json')

In [6]:
# Test the fine-tuned model
results = trainer.evaluate()
print(f"Fine-tuned model accuracy: {results['eval_accuracy']:.2%}")

# Test on a few examples
test_texts = [
    "This movie was amazing! Best film I've seen all year.",
    "Terrible waste of time. I want my money back.",
]

inputs = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt")
# Move input tensors to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

for text, pred in zip(test_texts, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Text: {text}")
    print(f"Prediction: {sentiment}\n")

Fine-tuned model accuracy: 84.00%
Text: This movie was amazing! Best film I've seen all year.
Prediction: Positive

Text: Terrible waste of time. I want my money back.
Prediction: Negative

